# Notebook 05：回测与评价

目标：从回测结果计算基准、超额收益、回撤和交易质量。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DB_PATH = ROOT / 'data' / 'stocks.db'
print(f'项目根目录: {ROOT}')
print(f'数据库: {DB_PATH}')

In [ ]:
import pandas as pd
from src.backtest import run_backtest_detailed
from src.report import calculate_report_metrics
from src.strategy import SmaCross

result = run_backtest_detailed(
    '000001',
    SmaCross,
    db_path=str(DB_PATH),
    strategy_params={'fast': 5, 'slow': 20},
)
metrics = calculate_report_metrics(result)
keys = ['total_return', 'benchmark_return', 'excess_return', 'annual_return', 'max_drawdown', 'sharpe_ratio', 'sortino_ratio', 'information_ratio', 'total_trades', 'win_rate']
pd.Series({k: metrics[k] for k in keys})

In [ ]:
trade_df = pd.DataFrame(result['trades'])
if not trade_df.empty:
    trade_df['return'] = trade_df['price_out'] / trade_df['price_in'] - 1
trade_df.head()

In [ ]:
equity = result['equity_curve'].copy()
equity['drawdown'] = equity['value'] / equity['value'].cummax() - 1
equity[['date', 'value', 'drawdown']].tail()

小结：报告扩展指标由 `src.report.calculate_report_metrics()` 统一计算，保证 Streamlit、脚本和 Notebook 口径一致。